# Solving Singular Systems of Linear Equations

*Course notes for **Math for Machine Learning**, C1 · W2 · L1 — "Solving Singular Systems of Linear Equations" (DeepLearning.AI).*

[Last time](./C1_W2_L1_solving_non_singular_systems.ipynb) elimination gave a unique answer because the system was non-singular. Here we run the *same* elimination on **singular** systems and watch what happens. The procedure doesn't break — it ends in one of two tell-tale signatures:

| Final line after elimination | Meaning |
|------------------------------|---------|
| $0 = 0$ | **redundant** → infinitely many solutions |
| $0 = (\text{non-zero})$ | **contradictory** → no solution |

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 1. Redundant system → infinitely many solutions

$$ \begin{cases} a + b = 10 \\ 2a + 2b = 20 \end{cases} $$

Divide each equation by its coefficient of $a$:

$$ a + b = 10 \qquad a + b = 10 $$

Both equations are now identical. Subtract equation 1 from equation 2:

$$ (a + b) - (a + b) = 10 - 10 \;\Rightarrow\; 0 = 0. $$

Always true, and carrying **no information**. We are left with a single equation $a + b = 10$. Pick $a$ freely; then $b = 10 - a$. **Infinitely many solutions.**

In [2]:
# Each equation: [coeff_a, coeff_b, rhs]
eq1 = np.array([1.0, 1.0, 10.0])
eq2 = np.array([2.0, 2.0, 20.0])

r1, r2 = eq1 / eq1[0], eq2 / eq2[0]   # normalise coefficient of a
print("after dividing:", r1, "and", r2)

eliminated = r2 - r1                    # subtract to eliminate a
print("eliminated row:", eliminated, " ->  0 = 0  (redundant)")

# A few valid solutions on the line a + b = 10:
for a in (0, 4, 10):
    print(f"  a = {a:2d},  b = {10 - a}")

after dividing: [ 1.  1. 10.] and [ 1.  1. 10.]
eliminated row: [0. 0. 0.]  ->  0 = 0  (redundant)
  a =  0,  b = 10
  a =  4,  b = 6
  a = 10,  b = 0


## 2. Contradictory system → no solution

$$ \begin{cases} a + b = 10 \\ 2a + 2b = 24 \end{cases} $$

Divide each by its coefficient of $a$:

$$ a + b = 10 \qquad a + b = 12 $$

Same left-hand sides, different right-hand sides. Subtract:

$$ 0 = 12 - 10 = 2. $$

$0 = 2$ is **never** true, so there is **no solution**.

In [3]:
eq1 = np.array([1.0, 1.0, 10.0])
eq2 = np.array([2.0, 2.0, 24.0])

r1, r2 = eq1 / eq1[0], eq2 / eq2[0]
print("after dividing:", r1, "and", r2)

eliminated = r2 - r1
print("eliminated row:", eliminated, f" ->  0 = {eliminated[2]:.0f}  (contradiction, no solution)")

after dividing: [ 1.  1. 10.] and [ 1.  1. 12.]
eliminated row: [0. 0. 2.]  ->  0 = 2  (contradiction, no solution)


## 3. Telling the two apart with rank

Both systems share the singular coefficient matrix $\begin{bmatrix}1&1\\2&2\end{bmatrix}$, so `np.linalg.solve` refuses both (it only handles non-singular matrices). To distinguish *infinite* from *none*, compare the rank of the coefficient matrix $A$ with the rank of the **augmented** matrix $[A \mid \mathbf{b}]$ (the Rouché–Capelli theorem):

- $\text{rank}(A) = \text{rank}([A\mid b]) <$ #unknowns → **infinitely many** solutions (redundant).
- $\text{rank}(A) < \text{rank}([A\mid b])$ → **no** solution (contradictory).

In [4]:
def classify_system(A, b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1, 1)
    n_unknowns = A.shape[1]
    rank_A = np.linalg.matrix_rank(A)
    rank_aug = np.linalg.matrix_rank(np.hstack([A, b]))
    if rank_A < rank_aug:
        return "no solution (contradictory)"
    if rank_A < n_unknowns:
        return "infinitely many solutions (redundant)"
    return "unique solution (non-singular)"

A = [[1, 1], [2, 2]]
print("redundant   2a+2b=20 :", classify_system(A, [10, 20]))
print("contradict. 2a+2b=24 :", classify_system(A, [10, 24]))

redundant   2a+2b=20 : infinitely many solutions (redundant)
contradict. 2a+2b=24 : no solution (contradictory)


## 4. Quiz

$$ \begin{cases} 5a + b = 11 \\ 10a + 2b = 22 \end{cases} $$

Dividing equation 2 by 2 gives exactly equation 1 — the equations are redundant, so the system has **infinitely many solutions**.

In [5]:
print(classify_system([[5, 1], [10, 2]], [11, 22]))
# Equation 2 is 2 x equation 1  ->  redundant  ->  infinitely many solutions,
# e.g. a = 0, b = 11;  a = 1, b = 6;  a = 2, b = 1; ...  all satisfy 5a + b = 11.
for a in (0, 1, 2):
    print(f"  a = {a},  b = {11 - 5 * a}")

infinitely many solutions (redundant)
  a = 0,  b = 11
  a = 1,  b = 6
  a = 2,  b = 1


## 5. Takeaways

- Elimination works on singular systems too — it just **collapses a row** to $0 = (\text{something})$.
- $0 = 0$ → the equations were **redundant** → a whole line/plane of solutions (**infinitely many**).
- $0 = (\text{non-zero})$ → the equations were **contradictory** → **no solution**.
- Either way the coefficient matrix is **singular** ($\det = 0$), so `np.linalg.solve` won't apply; use the **rank test** on $A$ vs $[A\mid b]$ to tell infinite from none.

*Companion notebooks:* [non-singular solving](./C1_W2_L1_solving_non_singular_systems.ipynb) · [geometry of singularity](./C1_W1_L1_geometric_notion_of_singularity.ipynb).